In [ ]:
import imp
import numpy as np
from pathlib import Path
from labdata.schema import *
from labdata import chronic_paper as cp
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# target dates for chronic timepoint comparison
from datetime import timedelta
COMPARISON_TIMEPOINTS = [timedelta(days=7), timedelta(days=28), timedelta(days=49), timedelta(days=70)]
SORT_PARAM = 5
CRITERIA_PARAM = 1

In [ ]:
chronic_insertions = cp.IBLMatchedInsertion()
chronic_recording_keys = []
for ins in chronic_insertions:
    recordings = (EphysRecording() & ins) * Session()

    for timepoint in COMPARISON_TIMEPOINTS:
        target_datetime = ins['procedure_datetime'] + timepoint
        recordings_with_time_diff = recordings.proj(days_from_target='ABS(TIMESTAMPDIFF(DAY, session_datetime, "{}"))'.format(target_datetime.strftime('%Y-%m-%d %H:%M:%S')))
        key = recordings_with_time_diff.fetch(order_by='days_from_target ASC', as_dict=True, limit=1)[0]
        if key['days_from_target'] > 3:
            print(f"{key['days_from_target']} away from target date {timepoint}")
        chronic_recording_keys.append(key)
chronic_probe_recordings = EphysRecording.proj() & chronic_recording_keys
acute_probe_recordings = EphysRecording.proj() & cp.IBLMatchedInsertion().to_ephys_session()

In [ ]:
# launch spike sorting for all recordings
all_probe_sessions =  (EphysRecording() & acute_probe_recordings.fetch('KEY') + chronic_probe_recordings.fetch('KEY')) - (SpikeSorting & dict(parameter_set_num=SORT_PARAM))
labdata_submission_commands = []
t = []
for k in all_probe_sessions.fetch(order_by='subject_name, session_name'):
    dd = f'labdata2 run spks -t aws -a {k["subject_name"]} -s {k["session_name"]} --force-submit -- -m ks4.0'
    t.append(dict(session_name=k['session_name'],
                   subject_name=k['subject_name'],))
    #dd = f'labdata2 run detect -t aws -a {k["subject_name"]} -s {k["session_name"]}'
    labdata_submission_commands.append(dd)
    #os.system(dd) # uncomment to run
labdata_submission_commands = np.unique(labdata_submission_commands)
print(f'There are {len(labdata_submission_commands)} sessions to process.')
print('\n'.join(labdata_submission_commands))

all_probe_session_keys = all_probe_sessions.fetch('KEY')

In [ ]:
# get the probe keys so we can run dredge on each probe-session
acute_session_probe_keys = EphysRecording.ProbeSetting() & (cp.IBLMatchedInsertion().EphysRecording().proj('session_name', 
                                                                                                           'dataset_name',
                                                                                                           chronic_mouse='subject_name',
                                                                                                           chronic_probe_id='probe_id',
                                                                                                           subject_name='matched_subject_name',
                                                                                                           probe_num='matched_probe_num'))
chronic_session_probe_keys = (EphysRecording.ProbeSetting() & chronic_probe_recordings)
chronic_probes = dj.U('subject_name', 'probe_num') & chronic_session_probe_keys # sessions have the same trajectories 

In [ ]:
acute_session_probe_keys
matched_acute = (cp.IBLMatchedInsertion().EphysRecording * (cp.IBLMatchedInsertion().EphysRecording.proj() & chronic_insertion)).proj('session_name',ppp='probe_id', temp='subject_name',subject_name='matched_subject_name', probe_num='matched_probe_num')
matched_acute * UnitCount & dict(parameter_set_num=SORT_PARAM,unit_criteria_id=CRITERIA_PARAM)

In [ ]:
UnitCount.populate()
ACUTE_DAYS_OFFSET = -10
#colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
colors = sns.color_palette("hls", len(chronic_probes))
fig, axs = plt.subplots(2,1, figsize=(10,10))

stats_rows = []
for i,c in enumerate(chronic_probes):
    chronic_insertion = ProbeInsertion * Session * UnitCount * (chronic_session_probe_keys & c) & dict(unit_criteria_id=CRITERIA_PARAM,
                                                                                                       parameter_set_num=SORT_PARAM)
    chronic_insertion = chronic_insertion.proj('sua','mua', days_from_insertion='DATEDIFF(session_datetime, procedure_datetime)')
    days, sua, mua = chronic_insertion.fetch('days_from_insertion','sua','mua', order_by='days_from_insertion ASC')
    axs[0].plot(days, sua, marker='o', color=colors[i])
    axs[1].plot(days, mua, marker='o', color=colors[i])
    for j,(s,m) in enumerate(zip(sua,mua)):
        stats_rows.append((i, 1, j, 0, m, s))

    # now do acute data
    matched_acute = (cp.IBLMatchedInsertion().EphysRecording * (cp.IBLMatchedInsertion().EphysRecording.proj() & chronic_insertion)).proj('session_name',ppp='probe_id', temp='subject_name',subject_name='matched_subject_name', probe_num='matched_probe_num')
    matched_acute = matched_acute * UnitCount & dict(parameter_set_num=SORT_PARAM,unit_criteria_id=CRITERIA_PARAM)
    acute_sua, acute_mua = matched_acute.fetch('sua','mua')
    offset = np.random.normal(scale=1, size=len(acute_sua)) + ACUTE_DAYS_OFFSET
    axs[0].scatter(offset, acute_sua, marker='o', color=colors[i], facecolors='none')
    axs[1].scatter(offset, acute_mua, marker='o', color=colors[i], facecolors='none')
    for j,(s,m) in enumerate(zip(acute_sua,acute_mua)):
        stats_rows.append((i, 0, 0, j, m, s))

xticks = [ACUTE_DAYS_OFFSET] + [c.days for c in COMPARISON_TIMEPOINTS]
xticklabels = ['Acute'] + [c.days for c in COMPARISON_TIMEPOINTS]
axs[0].set_xticks(xticks)
axs[0].set_xticklabels(xticklabels)
axs[1].set_xticks(xticks)
axs[1].set_xticklabels(xticklabels)
axs[0].set_ylabel('Num single units')
axs[1].set_ylabel('Num multi units')
axs[0].set_xlabel('Days post insertion')
axs[1].set_xlabel('Days post insertion')

stats_table = pd.DataFrame(stats_rows, columns=['insertion_site','is_chronic','timepoint','session_num','multi_units','single_units'])
stats_table.to_csv(Path().resolve().parent / 'stats_tables' / 'acute_chronic_unit_count.csv', index=False)